# Logistic Regression from Scratch — Telco Customer Churn

**Purpose:** This notebook is for the Logistic Regression part of the AI3013 Machine Learning final project.  
It uses the processed files generated by the preprocessing member:

- `train_processed.csv`
- `test_processed.csv`
- `train_processed_oversampled.csv`
- `information_gain_ranking.csv` (optional explanation only)

## Main idea
This notebook does not simply train a standard Logistic Regression model. It includes a project-level novelty:

> **Cost-sensitive Logistic Regression with validation-based threshold tuning for churn detection.**

Because customer churn is an imbalanced and business-sensitive problem, missing a real churn customer is usually more costly than incorrectly flagging a non-churn customer. Therefore, this notebook compares:

1. Standard Logistic Regression on the original training data.
2. Logistic Regression trained on the oversampled training data.
3. Class-weighted Logistic Regression.
4. **Novelty version:** class-weighted Logistic Regression + threshold selected from the validation set to prioritize churn recall / F2-score.

No `sklearn` model or metric is used. The Logistic Regression, metrics, stratified split, cross-validation, and threshold search are implemented from scratch using `numpy` and `pandas`.

## 1. Imports and output folders

In [ ]:
from pathlib import Path
import time
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Output folders
OUTPUT_DIR = Path("outputs_lr")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Outputs will be saved to: {OUTPUT_DIR.resolve()}")
print(f"Figures will be saved to: {FIG_DIR.resolve()}")

## 2. Load processed datasets

The code below is written to work in different folder structures. It searches for the processed CSV files in:

1. the current notebook folder,
2. `outputs/processed_data/`,
3. `/mnt/data/` if running inside ChatGPT's sandbox.

In [ ]:
def find_file(filename: str) -> Path:
    candidates = [
        Path(filename),
        Path("outputs") / "processed_data" / filename,
        Path("code") / "outputs" / "processed_data" / filename,
        Path("/mnt/data") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Cannot find {filename}. Put it in the same folder as this notebook or in outputs/processed_data/."
    )

train_path = find_file("train_processed.csv")
test_path = find_file("test_processed.csv")
oversampled_path = find_file("train_processed_oversampled.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
oversampled_df = pd.read_csv(oversampled_path)

print("Loaded files:")
print("Train:", train_path, train_df.shape)
print("Test:", test_path, test_df.shape)
print("Oversampled train:", oversampled_path, oversampled_df.shape)

assert "Churn" in train_df.columns, "Target column Churn is missing in train_processed.csv"
assert list(train_df.columns) == list(test_df.columns), "Train and test columns are not identical"
assert list(train_df.columns) == list(oversampled_df.columns), "Train and oversampled columns are not identical"

feature_cols = [c for c in train_df.columns if c != "Churn"]

X_train = train_df[feature_cols].to_numpy(dtype=float)
y_train = train_df["Churn"].to_numpy(dtype=int)

X_test = test_df[feature_cols].to_numpy(dtype=float)
y_test = test_df["Churn"].to_numpy(dtype=int)

X_train_over = oversampled_df[feature_cols].to_numpy(dtype=float)
y_train_over = oversampled_df["Churn"].to_numpy(dtype=int)

print("\nClass distribution:")
print("Original train:", pd.Series(y_train).value_counts().sort_index().to_dict())
print("Test:", pd.Series(y_test).value_counts().sort_index().to_dict())
print("Oversampled train:", pd.Series(y_train_over).value_counts().sort_index().to_dict())

## 3. Utility functions: metrics, split, and plotting

All metrics are implemented manually so that the evaluation is consistent with the from-scratch requirement.

In [ ]:
def confusion_matrix_binary(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    return {"TN": tn, "FP": fp, "FN": fn, "TP": tp}


def safe_divide(a, b):
    return 0.0 if b == 0 else a / b


def classification_metrics(y_true, y_pred, positive_label=1, fn_cost=5.0, fp_cost=1.0):
    cm = confusion_matrix_binary(y_true, y_pred)
    tn, fp, fn, tp = cm["TN"], cm["FP"], cm["FN"], cm["TP"]
    accuracy = safe_divide(tp + tn, tp + tn + fp + fn)
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    f1 = safe_divide(2 * precision * recall, precision + recall)
    beta = 2.0
    f2 = safe_divide((1 + beta**2) * precision * recall, beta**2 * precision + recall)
    balanced_accuracy = (recall + specificity) / 2
    business_cost = fn_cost * fn + fp_cost * fp
    return {
        **cm,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "f2": f2,
        "balanced_accuracy": balanced_accuracy,
        "business_cost": business_cost,
    }


def stratified_split_indices(y, test_size=0.2, random_state=42):
    rng = np.random.default_rng(random_state)
    y = np.asarray(y)
    train_idx, test_idx = [], []
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        idx = rng.permutation(idx)
        n_test = int(round(len(idx) * test_size))
        test_idx.extend(idx[:n_test].tolist())
        train_idx.extend(idx[n_test:].tolist())
    return np.array(train_idx), np.array(test_idx)


def make_class_weights(y):
    # Balanced class weights: n_samples / (n_classes * class_count).
    y = np.asarray(y)
    counts = {int(label): int(np.sum(y == label)) for label in np.unique(y)}
    n = len(y)
    k = len(counts)
    return {label: n / (k * count) for label, count in counts.items()}


def estimate_memory_mb(X, y, model=None):
    total_bytes = X.nbytes + y.nbytes
    if model is not None:
        total_bytes += model.weights.nbytes + np.array([model.bias]).nbytes
    return total_bytes / (1024 ** 2)


def print_metrics_table(df):
    display_cols = [
        "model", "threshold", "accuracy", "precision", "recall", "f1", "f2",
        "balanced_accuracy", "business_cost", "TN", "FP", "FN", "TP",
        "training_time_sec", "prediction_time_sec", "memory_mb"
    ]
    existing = [c for c in display_cols if c in df.columns]
    return df[existing].sort_values(["f2", "recall"], ascending=False)

## 4. Logistic Regression from scratch

The model uses:

- sigmoid function,
- binary cross-entropy loss,
- optional L2 regularization,
- optional class weighting for imbalanced churn detection,
- full-batch gradient descent.

The weighted version is part of the novelty because the loss function gives higher importance to churn customers (`Churn = 1`).

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.05, epochs=3000, l2_lambda=0.001, class_weight=None, verbose=False):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.l2_lambda = l2_lambda
        self.class_weight = class_weight
        self.verbose = verbose
        self.weights = None
        self.bias = 0.0
        self.loss_history = []

    @staticmethod
    def sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def _sample_weights(self, y):
        if self.class_weight is None:
            return np.ones_like(y, dtype=float)
        return np.array([self.class_weight[int(label)] for label in y], dtype=float)

    def compute_loss(self, X, y):
        n = X.shape[0]
        p = self.predict_proba(X)
        eps = 1e-15
        p = np.clip(p, eps, 1 - eps)
        sw = self._sample_weights(y)
        data_loss = -np.mean(sw * (y * np.log(p) + (1 - y) * np.log(1 - p)))
        reg_loss = (self.l2_lambda / (2 * n)) * np.sum(self.weights ** 2)
        return float(data_loss + reg_loss)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features, dtype=float)
        self.bias = 0.0
        self.loss_history = []
        sw = self._sample_weights(y)

        for epoch in range(self.epochs):
            linear_output = X @ self.weights + self.bias
            y_prob = self.sigmoid(linear_output)
            error = (y_prob - y) * sw

            grad_w = (X.T @ error) / n_samples + (self.l2_lambda / n_samples) * self.weights
            grad_b = np.mean(error)

            self.weights -= self.learning_rate * grad_w
            self.bias -= self.learning_rate * grad_b

            if epoch % 20 == 0 or epoch == self.epochs - 1:
                self.loss_history.append(self.compute_loss(X, y))

            if self.verbose and epoch % 500 == 0:
                print(f"Epoch {epoch:4d} | loss = {self.loss_history[-1]:.5f}")
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        return self.sigmoid(X @ self.weights + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

## 5. Training helper

This function trains a model, evaluates it on the untouched test set, and records time/memory.  
The test set is **never** used for threshold selection.

In [ ]:
def train_and_evaluate(model_name, X_tr, y_tr, X_te, y_te, class_weight=None, threshold=0.5,
                       learning_rate=0.05, epochs=3000, l2_lambda=0.001):
    model = LogisticRegressionScratch(
        learning_rate=learning_rate,
        epochs=epochs,
        l2_lambda=l2_lambda,
        class_weight=class_weight,
        verbose=False,
    )

    start_train = time.perf_counter()
    model.fit(X_tr, y_tr)
    training_time = time.perf_counter() - start_train

    start_pred = time.perf_counter()
    y_pred = model.predict(X_te, threshold=threshold)
    y_prob = model.predict_proba(X_te)
    prediction_time = time.perf_counter() - start_pred

    metrics = classification_metrics(y_te, y_pred)
    metrics.update({
        "model": model_name,
        "threshold": threshold,
        "training_time_sec": training_time,
        "prediction_time_sec": prediction_time,
        "memory_mb": estimate_memory_mb(X_tr, y_tr, model),
        "learning_rate": learning_rate,
        "epochs": epochs,
        "l2_lambda": l2_lambda,
    })
    return model, y_prob, metrics

## 6. Baseline experiments

The first three models keep the default threshold at `0.5`:

1. **Baseline LR:** trained on the original processed training data.
2. **Oversampled LR:** trained on the oversampled training data.
3. **Class-weighted LR:** trained on the original training data but with positive churn class receiving a larger weight.

In [ ]:
class_weights = make_class_weights(y_train)
print("Class weights for weighted Logistic Regression:", class_weights)

experiments = []
models = {}
probs = {}

baseline_model, baseline_prob, baseline_metrics = train_and_evaluate(
    "LR_baseline_original_train_t0.50",
    X_train, y_train, X_test, y_test,
    class_weight=None,
    threshold=0.5,
    learning_rate=0.05,
    epochs=3000,
    l2_lambda=0.001,
)
experiments.append(baseline_metrics)
models[baseline_metrics["model"]] = baseline_model
probs[baseline_metrics["model"]] = baseline_prob

oversampled_model, oversampled_prob, oversampled_metrics = train_and_evaluate(
    "LR_oversampled_train_t0.50",
    X_train_over, y_train_over, X_test, y_test,
    class_weight=None,
    threshold=0.5,
    learning_rate=0.05,
    epochs=3000,
    l2_lambda=0.001,
)
experiments.append(oversampled_metrics)
models[oversampled_metrics["model"]] = oversampled_model
probs[oversampled_metrics["model"]] = oversampled_prob

weighted_model, weighted_prob, weighted_metrics = train_and_evaluate(
    "LR_class_weighted_t0.50",
    X_train, y_train, X_test, y_test,
    class_weight=class_weights,
    threshold=0.5,
    learning_rate=0.05,
    epochs=3000,
    l2_lambda=0.001,
)
experiments.append(weighted_metrics)
models[weighted_metrics["model"]] = weighted_model
probs[weighted_metrics["model"]] = weighted_prob

results_df = pd.DataFrame(experiments)
print_metrics_table(results_df)

## 7. Novelty experiment: validation-based threshold tuning

### Why this is needed
A default threshold of `0.5` assumes that false positives and false negatives have similar consequences. In churn prediction, this is not realistic:

- **False negative:** the model fails to identify a customer who will actually churn. The company may lose this customer.
- **False positive:** the model flags a customer who will not churn. The company may spend extra retention effort.

Usually, a false negative is more costly. Therefore, this notebook chooses a decision threshold from the validation set to improve churn-focused metrics, especially **F2-score**, where recall is weighted more heavily than precision.

This is the main novelty of the Logistic Regression part:  
**not a new theoretical model, but a churn-oriented decision strategy built on from-scratch Logistic Regression.**

In [ ]:
# Create a validation set from the original training set only.
inner_train_idx, val_idx = stratified_split_indices(y_train, test_size=0.2, random_state=RANDOM_STATE)
X_inner_train, y_inner_train = X_train[inner_train_idx], y_train[inner_train_idx]
X_val, y_val = X_train[val_idx], y_train[val_idx]

inner_class_weights = make_class_weights(y_inner_train)
print("Inner train shape:", X_inner_train.shape)
print("Validation shape:", X_val.shape)
print("Inner class weights:", inner_class_weights)

threshold_model = LogisticRegressionScratch(
    learning_rate=0.05,
    epochs=3000,
    l2_lambda=0.001,
    class_weight=inner_class_weights,
)
threshold_model.fit(X_inner_train, y_inner_train)
val_prob = threshold_model.predict_proba(X_val)

threshold_rows = []
for threshold in np.arange(0.05, 0.951, 0.01):
    val_pred = (val_prob >= threshold).astype(int)
    m = classification_metrics(y_val, val_pred)
    m["threshold"] = round(float(threshold), 2)
    threshold_rows.append(m)

threshold_df = pd.DataFrame(threshold_rows)

# Choose the threshold that maximizes F2-score. Tie-breaker: lower business cost, then higher recall.
best_row = threshold_df.sort_values(
    by=["f2", "business_cost", "recall"],
    ascending=[False, True, False]
).iloc[0]

best_threshold = float(best_row["threshold"])
print("Best threshold selected from validation set:", best_threshold)
print(best_row[["threshold", "precision", "recall", "f1", "f2", "business_cost", "TN", "FP", "FN", "TP"]])

threshold_df.to_csv(OUTPUT_DIR / "lr_threshold_search_validation.csv", index=False)

## 8. Final novelty model on the untouched test set

After selecting the threshold using only the validation set, we train the class-weighted model on the full original training set and apply the selected threshold to the untouched test set.

In [ ]:
novel_model, novel_prob, novel_metrics = train_and_evaluate(
    f"LR_novel_weighted_threshold_t{best_threshold:.2f}",
    X_train, y_train, X_test, y_test,
    class_weight=class_weights,
    threshold=best_threshold,
    learning_rate=0.05,
    epochs=3000,
    l2_lambda=0.001,
)
experiments.append(novel_metrics)
models[novel_metrics["model"]] = novel_model
probs[novel_metrics["model"]] = novel_prob

results_df = pd.DataFrame(experiments)
results_df.to_csv(OUTPUT_DIR / "lr_all_experiments.csv", index=False)

# This file can be merged into the group's final results_summary.csv.
lr_metrics_df = results_df.copy()
lr_metrics_df.to_csv(OUTPUT_DIR / "lr_metrics.csv", index=False)

print_metrics_table(results_df)

## 9. Manual 5-fold cross-validation

This gives a more stable view of Logistic Regression performance and can be reported briefly in the result section.  
The cross-validation is also implemented manually.

In [ ]:
def stratified_kfold_indices(y, n_splits=5, random_state=42):
    rng = np.random.default_rng(random_state)
    y = np.asarray(y)
    folds = [[] for _ in range(n_splits)]
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        idx = rng.permutation(idx)
        for i, sample_idx in enumerate(idx):
            folds[i % n_splits].append(int(sample_idx))
    all_indices = set(range(len(y)))
    split_indices = []
    for fold in folds:
        val_idx = np.array(sorted(fold))
        train_idx = np.array(sorted(list(all_indices - set(val_idx))))
        split_indices.append((train_idx, val_idx))
    return split_indices


def run_cv(X, y, model_label, class_weight_mode=None, threshold=0.5, n_splits=5):
    rows = []
    for fold_id, (tr_idx, va_idx) in enumerate(stratified_kfold_indices(y, n_splits=n_splits, random_state=RANDOM_STATE), start=1):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_va, y_va = X[va_idx], y[va_idx]
        cw = make_class_weights(y_tr) if class_weight_mode == "balanced" else None
        model = LogisticRegressionScratch(
            learning_rate=0.05,
            epochs=2000,
            l2_lambda=0.001,
            class_weight=cw,
        )
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_va, threshold=threshold)
        m = classification_metrics(y_va, y_pred)
        m.update({"model": model_label, "fold": fold_id, "threshold": threshold})
        rows.append(m)
    return pd.DataFrame(rows)

cv_baseline = run_cv(X_train, y_train, "CV_LR_baseline", class_weight_mode=None, threshold=0.5, n_splits=5)
cv_weighted = run_cv(X_train, y_train, "CV_LR_weighted", class_weight_mode="balanced", threshold=0.5, n_splits=5)
cv_results = pd.concat([cv_baseline, cv_weighted], ignore_index=True)
cv_results.to_csv(OUTPUT_DIR / "lr_cv_results.csv", index=False)

cv_summary = cv_results.groupby("model")[["accuracy", "precision", "recall", "f1", "f2", "balanced_accuracy", "business_cost"]].agg(["mean", "std"])
cv_summary.to_csv(OUTPUT_DIR / "lr_cv_summary.csv")
cv_summary

## 10. Figures

The following figures are saved into `outputs_lr/figures/` and can be used directly in the report.

In [ ]:
# Figure 1: Loss curves
plt.figure(figsize=(9, 5.5))
for name, model in models.items():
    x_axis = np.arange(len(model.loss_history)) * 20
    if len(x_axis) > len(model.loss_history):
        x_axis = x_axis[:len(model.loss_history)]
    plt.plot(x_axis, model.loss_history, label=name.replace("LR_", ""))
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Logistic Regression Training Loss Curves")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "lr_loss_curves.png", dpi=300)
plt.show()

# Figure 2: Threshold sweep on validation set
plt.figure(figsize=(9, 5.5))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1-score")
plt.plot(threshold_df["threshold"], threshold_df["f2"], label="F2-score")
plt.axvline(best_threshold, linestyle="--", label=f"Selected threshold = {best_threshold:.2f}")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Validation Threshold Search for Churn-Oriented Logistic Regression")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "lr_threshold_sweep_validation.png", dpi=300)
plt.show()

# Figure 3: Model comparison bar chart
plot_df = results_df.set_index("model")[["accuracy", "precision", "recall", "f1", "f2", "balanced_accuracy"]]
ax = plot_df.plot(kind="bar", figsize=(11, 6))
ax.set_title("Logistic Regression Experiment Comparison on Test Set")
ax.set_ylabel("Score")
ax.set_xlabel("Experiment")
ax.set_ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.legend(loc="lower right", fontsize=8)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "lr_model_comparison_bar.png", dpi=300)
plt.show()

In [ ]:
def plot_confusion_matrix(cm, title, ax):
    matrix = np.array([[cm["TN"], cm["FP"]], [cm["FN"], cm["TP"]]])
    im = ax.imshow(matrix)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred 0", "Pred 1"])
    ax.set_yticklabels(["True 0", "True 1"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=12)
    return im

# Figure 4: Confusion matrices for all experiments
fig, axes = plt.subplots(1, len(results_df), figsize=(4 * len(results_df), 4))
if len(results_df) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, results_df.iterrows()):
    cm = {k: int(row[k]) for k in ["TN", "FP", "FN", "TP"]}
    title = row["model"].replace("LR_", "").replace("_", "\n")
    plot_confusion_matrix(cm, title, ax)
plt.suptitle("Confusion Matrices on Test Set", y=1.05)
plt.tight_layout()
plt.savefig(FIG_DIR / "lr_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 5: Top coefficients from the final novelty model
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": novel_model.weights,
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
coef_df.to_csv(OUTPUT_DIR / "lr_coefficients_novel_model.csv", index=False)

top_n = 15
top_coef = coef_df.head(top_n).sort_values("coefficient")
plt.figure(figsize=(9, 6))
plt.barh(top_coef["feature"], top_coef["coefficient"])
plt.axvline(0, linewidth=1)
plt.xlabel("Coefficient")
plt.title(f"Top {top_n} Logistic Regression Coefficients — Novel Weighted Model")
plt.tight_layout()
plt.savefig(FIG_DIR / "lr_top_coefficients_novel_model.png", dpi=300)
plt.show()

# Figure 6: Predicted probability distribution by actual class for the final novelty model
plt.figure(figsize=(9, 5.5))
plt.hist(novel_prob[y_test == 0], bins=30, alpha=0.65, label="Actual No Churn (0)")
plt.hist(novel_prob[y_test == 1], bins=30, alpha=0.65, label="Actual Churn (1)")
plt.axvline(best_threshold, linestyle="--", label=f"Selected threshold = {best_threshold:.2f}")
plt.xlabel("Predicted Churn Probability")
plt.ylabel("Number of Customers")
plt.title("Predicted Churn Probability Distribution on Test Set")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "lr_probability_distribution_novel_model.png", dpi=300)
plt.show()

## 11. Optional: information gain ranking for explanation only

Information gain was computed by the preprocessing member. Here we only load it to support the interpretation of important churn-related features.  
Do **not** use the full-data information gain ranking for feature selection in the main model, because that could create data leakage if the ranking was computed before the train/test split.

In [ ]:
try:
    ig_path = find_file("information_gain_ranking.csv")
    ig_df = pd.read_csv(ig_path)
    display(ig_df.head(10))

    plt.figure(figsize=(8, 5))
    top_ig = ig_df.head(10).sort_values("information_gain")
    plt.barh(top_ig["feature"], top_ig["information_gain"])
    plt.xlabel("Information Gain")
    plt.title("Top Information Gain Features from Preprocessing Output")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "lr_information_gain_top_features.png", dpi=300)
    plt.show()
except Exception as e:
    print("Information gain ranking was not loaded:", e)

## 12. Save a group-ready summary row

The group's final `results_summary.csv` should contain one row for the Logistic Regression result you decide to report.  
Recommended: report the novelty version if it improves recall/F2 and explain the precision trade-off.

In [ ]:
# Choose recommended LR row: highest F2, with recall as tie-breaker.
recommended_row = results_df.sort_values(["f2", "recall", "balanced_accuracy"], ascending=False).iloc[[0]].copy()
recommended_row.insert(0, "algorithm", "Logistic Regression")
recommended_row.to_csv(OUTPUT_DIR / "lr_recommended_result_for_results_summary.csv", index=False)

print("Recommended Logistic Regression result for group comparison:")
display(recommended_row)

print("\nSaved files:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", path)
print("\nSaved figures:")
for path in sorted(FIG_DIR.glob("*.png")):
    print("-", path)

## 13. How to explain the novelty in Q&A

### Novelty statement
This project applies a common model, Logistic Regression, to a well-known churn prediction dataset. Therefore, the novelty is not claiming that Logistic Regression itself is new. The project-level novelty is:

> A from-scratch, churn-oriented Logistic Regression pipeline that combines class-weighted learning with validation-based threshold tuning to reduce missed churn customers.

### Why it is reasonable
Standard Logistic Regression uses a 0.5 threshold. However, in churn prediction, the minority class (`Churn = 1`) is more important for business decisions. A model with high overall accuracy may still miss many churn customers. The threshold tuning step changes the final decision rule based on validation performance, especially F2-score and business cost.

### What the teacher may ask

**Q1: Is this really a new model?**  
No. The base model is still Logistic Regression. The novelty is an adapted method for the churn context: cost-sensitive loss + threshold tuning. This is a project-level methodological improvement, not a new theoretical algorithm.

**Q2: Why use F2-score instead of only accuracy?**  
Accuracy can be misleading when churn customers are the minority. F2-score gives more weight to recall, which is useful because missing a true churn customer is usually more costly than sending a retention offer to a non-churn customer.

**Q3: Why not tune the threshold on the test set?**  
That would be data leakage. The threshold is selected using a validation set split from the training data only. The test set remains untouched until final evaluation.

**Q4: Why compare with oversampling?**  
Oversampling is a common imbalance-handling technique. Comparing it with class weighting helps show whether changing the training distribution or changing the loss function is more suitable for this dataset.

**Q5: Why use L2 regularization?**  
The processed data uses one-hot encoding with all dummy columns retained, so some features may be correlated. L2 regularization stabilizes the weights and reduces overfitting risk.

## Extra evidence for presentation: threshold comparison on test set

This cell is only for reporting and presentation evidence. The threshold was selected on the validation set, not on the test set. Here, the test-set table simply shows the trade-off after the selected threshold is fixed. It helps explain why a lower threshold is more suitable when the project goal is to reduce missed churn customers.

In [ ]:
# Extra threshold comparison for report / presentation
# Important: best_threshold was already selected using the validation set above.
# This test-set table is for explanation only, not for choosing the threshold.
selected_thresholds = [0.20, 0.25, 0.30, 0.35, 0.36, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

test_threshold_rows = []
for threshold in selected_thresholds:
    y_pred_t = (novel_prob >= threshold).astype(int)
    m = classification_metrics(y_test, y_pred_t)
    m["threshold"] = threshold
    test_threshold_rows.append(m)

test_threshold_df = pd.DataFrame(test_threshold_rows)
test_threshold_df = test_threshold_df[[
    "threshold", "accuracy", "precision", "recall", "f1", "f2", "balanced_accuracy",
    "business_cost", "TN", "FP", "FN", "TP"
]]

test_threshold_df.to_csv(OUTPUT_DIR / "lr_threshold_comparison_test_selected.csv", index=False)
print("Test-set threshold comparison saved to:", OUTPUT_DIR / "lr_threshold_comparison_test_selected.csv")
display(test_threshold_df)

# Figure: score trade-off across thresholds on the test set
plt.figure(figsize=(9, 5.5))
for metric in ["precision", "recall", "f1", "f2"]:
    plt.plot(test_threshold_df["threshold"], test_threshold_df[metric], marker="o", label=metric.upper() if metric in ["f1", "f2"] else metric.capitalize())
plt.axvline(best_threshold, linestyle="--", label=f"Selected threshold = {best_threshold:.2f}")
plt.xlabel("Decision Threshold")
plt.ylabel("Score on Test Set")
plt.title("Threshold Trade-off for Class-weighted Logistic Regression")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "lr_threshold_comparison_test_selected.png", dpi=300)
plt.show()

# Figure: business-oriented error trade-off
plt.figure(figsize=(9, 5.5))
plt.plot(test_threshold_df["threshold"], test_threshold_df["FN"], marker="o", label="False negatives (missed churners)")
plt.plot(test_threshold_df["threshold"], test_threshold_df["FP"], marker="o", label="False positives")
plt.plot(test_threshold_df["threshold"], test_threshold_df["business_cost"], marker="o", label="Business cost = 5*FN + FP")
plt.axvline(best_threshold, linestyle="--", label=f"Selected threshold = {best_threshold:.2f}")
plt.xlabel("Decision Threshold")
plt.ylabel("Count / Cost")
plt.title("Business-oriented Error Trade-off Across Thresholds")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "lr_threshold_error_cost_test_selected.png", dpi=300)
plt.show()
